# Exercise — Run Governance as an Operation

Governance is ongoing, not a one-time project. **Trailhead Provisions** gives you a single
audit that rolls up every governance signal — missing catalog metadata, quality failures,
lineage gaps, consent conflict, and **access-control gaps**. Your job: run it, **remediate what
is fixable in AWS (the catalog in Glue and access in Lake Formation), and re-run until the
platform is compliant** — then write up what remains as ongoing watch items. See `INSTRUCTIONS.md`.

> Runs against live AWS Glue + Lake Formation when provisioned; local fallback offline.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog("trailhead.db")
print("Backend:", lf_backend(), "->", type(gc).__name__)

## 1. Run the platform governance audit (provided)

In [ ]:
findings = gc.governance_audit(contracts_valid=True)
print("Total findings:", len(findings))
findings.groupby(["signal", "severity"]).size().reset_index(name="count")

## 2. Remediate the catalog gaps in Glue
Close every `missing_metadata` finding by writing owner / classification / retention and tagging untagged columns (PII where the column holds personal data).

In [ ]:
PII_COLS = {"email", "phone", "cust_ref", "full_name"}
for tbl in gc.tables():
    tm = gc.catalog.table(tbl)
    gc.set_table_metadata(
        tbl,
        owner=None if tm.owner else "data-governance@trailhead.example",
        classification=None if tm.classification else "SENSITIVE",
        retention=None if tm.retention else "P5Y",
    )
    for col in tm.columns:
        if col.classification is None:
            gc.tag_column(tbl, col.name, "PII" if col.name in PII_COLS else "PUBLIC")
print("Catalog gaps remaining:", len(gc.catalog_audit()))

## 3. Remediate the access-control gaps in Lake Formation
Every **PII table** must be governed by Lake Formation. Bring each PII-bearing table under an LF-Tag and grant the steward by tag.

In [ ]:
for tbl in gc.tables():
    tm = gc.catalog.table(tbl)
    if tm.classification == "PII" or any(c.classification == "PII" for c in tm.columns):
        gc.assign_tag(tbl, "governed", "true")
gc.grant_by_tag("co_data_steward", "governed", "true")
print("Access governed for the PII tables.")

## 4. Re-audit and confirm compliance (provided)
The AWS-fixable findings — `missing_metadata` and `access_control_gap` — should both be zero. What remains are the recurring SLOs (quality, consent, lineage).

In [ ]:
re_findings = gc.governance_audit(contracts_valid=True)
def n(df, sig): return int((df["signal"] == sig).sum())
print("missing_metadata: ", n(findings, "missing_metadata"),  "->", n(re_findings, "missing_metadata"))
print("access_control_gap:", n(findings, "access_control_gap"), "->", n(re_findings, "access_control_gap"))
compliant = n(re_findings, "missing_metadata") == 0 and n(re_findings, "access_control_gap") == 0
print("AWS-fixable findings cleared?", compliant)   # True once you have remediated above
re_findings.groupby(["signal", "severity"]).size().reset_index(name="count")

## 5. Operations write-up
Replace the cell below with your write-up. Address every requirement in `INSTRUCTIONS.md`.

### Governance operations write-up

**Drove the platform to compliance in two control planes:**
- **Catalog (Glue):** closed every `missing_metadata` finding — set `owner`, `classification`,
  and `retention` on the tables that lacked them, and tagged untagged columns, classifying
  personal-data columns (`email`, `cust_ref`, etc.) as **PII**. The re-audit shows zero catalog gaps.
- **Access (Lake Formation):** classifying those PII columns made their tables subject to access
  governance, which the audit then flagged. I brought every PII table under an `governed=true`
  LF-Tag with a steward grant, taking `access_control_gap` to zero.

**What remains — ongoing SLOs, not one-time fixes:**
- **Quality violations** across the six dimensions — these recur every load; they belong on a
  monitored dashboard with per-dimension thresholds and domain owners, not a one-time cleanup.
- **Consent conflict** — loyalty and marketing keep disagreeing consent records; the durable fix
  is one consent system of record, tracked until that re-platforming lands.
- **Lineage gaps** — tables with no lineage events; Platform Engineering instruments them.

**Operating model:** run this audit on a schedule, auto-remediate or ticket the catalog/access
findings, and trend the SLO signals — governance is the loop, not the cleanup.